In [3]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
import implicit
from implicit.als import AlternatingLeastSquares
from implicit.nearest_neighbours import CosineRecommender
from tqdm import tqdm

# 1. Загрузка и фильтрация (оставляем только лайки >= 4)
url = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_v2/categoryFilesSmall/Digital_Music_5.json.gz"
df = pd.read_json(url, lines=True, compression='gzip')
df = df[['reviewerID', 'asin', 'overall', 'unixReviewTime']]
df.columns = ['user_id', 'item_id', 'rating', 'timestamp']
df = df[df['rating'] >= 4].copy()

# 2. Фильтр >= 2 отзывов (после отсечения оценок 1-3 могли появиться юзеры с 1 отзывом)
user_counts = df['user_id'].value_counts()
df = df[df['user_id'].isin(user_counts[user_counts >= 2].index)]

# 3. Leave-one-out
df = df.sort_values(by=['user_id', 'timestamp'])
test_df = df.groupby('user_id').tail(1)
train_df = df.drop(test_df.index)

# 4. Маппинги
all_users = train_df['user_id'].unique()
all_items = train_df['item_id'].unique()
user_to_index = {u: i for i, u in enumerate(all_users)}
item_to_index = {item: i for i, item in enumerate(all_items)}
index_to_item = {i: item for item, i in item_to_index.items()}

train_df['user_idx'] = train_df['user_id'].map(user_to_index)
train_df['item_idx'] = train_df['item_id'].map(item_to_index)

# Очищаем тест от товаров, которых нет в трейне
test_df = test_df[test_df['item_id'].isin(item_to_index)]

# 5. ИСПРАВЛЕНИЕ: Создаем ГЛАВНУЮ матрицу user-item (строки - юзеры, столбцы - товары)
# Именно её требует современная версия implicit!
sparse_user_item = csr_matrix(
    (np.ones(len(train_df), dtype=np.float32), 
     (train_df['user_idx'].astype(np.int32), train_df['item_idx'].astype(np.int32))),
    shape=(len(all_users), len(all_items))
).tocsr()

# 6. Обучение моделей
popular_items = train_df['item_id'].value_counts().index.tolist()

print("Обучение Cosine Recommender...")
# Для item-item CF передаем матрицу user-item
model_cosine = CosineRecommender(K=50)
model_cosine.fit(sparse_user_item, show_progress=False)

print("Обучение ALS...")
model_als = AlternatingLeastSquares(factors=64, regularization=0.05, iterations=20, use_gpu=False)
model_als.fit(sparse_user_item, show_progress=False)

# 7. Функции рекомендаций
def recommend_popular(user_id, k):
    user_history = set(train_df[train_df['user_id'] == user_id]['item_id'].tolist())
    recs = [item for item in popular_items if item not in user_history]
    return recs[:k]

def recommend_cosine(user_id, k):
    if user_id not in user_to_index: return recommend_popular(user_id, k)
    user_idx = user_to_index[user_id]
    # Передаем строку юзера из ПРАВИЛЬНОЙ матрицы
    ids, scores = model_cosine.recommend(user_idx, sparse_user_item[user_idx], N=k, filter_already_liked_items=True)
    return [index_to_item[int(i)] for i in ids if int(i) in index_to_item]

def recommend_als(user_id, k):
    if user_id not in user_to_index: return recommend_popular(user_id, k)
    user_idx = user_to_index[user_id]
    # Передаем строку юзера из ПРАВИЛЬНОЙ матрицы
    ids, scores = model_als.recommend(user_idx, sparse_user_item[user_idx], N=k, 
                                      filter_already_liked_items=True, recalculate_user=True)
    return [index_to_item[int(i)] for i in ids if int(i) in index_to_item]

# 8. Оценка
def evaluate_model(model_name, recommend_func, k=10):
    hits, mrr_sum, ndcg_sum = 0, 0.0, 0.0
    recommended_items_set = set()
    total_users = len(test_df)
    catalog_size = len(item_to_index)

    for _, row in tqdm(test_df.iterrows(), total=total_users, desc=f"Evaluating {model_name}"):
        user_id = row['user_id']
        true_item = row['item_id']
        rec_items = recommend_func(user_id, k)
        recommended_items_set.update(rec_items)
        
        if true_item in rec_items:
            hits += 1
            rank = rec_items.index(true_item) + 1
            mrr_sum += 1.0 / rank
            ndcg_sum += 1.0 / np.log2(rank + 1)
            
    print(f"\n--- {model_name} ---")
    print(f"HR@{k}:   {hits / total_users:.4f}")
    print(f"MRR@{k}:  {mrr_sum / total_users:.4f}")
    print(f"NDCG@{k}: {ndcg_sum / total_users:.4f}")
    print(f"Cov:      {len(recommended_items_set) / catalog_size:.4f}\n")

# Запуск
evaluate_model("Top Popular", recommend_popular)
evaluate_model("Cosine", recommend_cosine)
evaluate_model("ALS", recommend_als)

Обучение Cosine Recommender...
Обучение ALS...


/Users/bot/Desktop/Study/ML_Level_2/venv/lib/python3.12/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.0005900859832763672 seconds
  warnings.warn(
Evaluating Top Popular: 100%|██████████| 16356/16356 [00:12<00:00, 1327.67it/s]



--- Top Popular ---
HR@10:   0.0109
MRR@10:  0.0050
NDCG@10: 0.0063
Cov:      0.0013



Evaluating Cosine: 100%|██████████| 16356/16356 [00:00<00:00, 27793.41it/s]



--- Cosine ---
HR@10:   0.0820
MRR@10:  0.0504
NDCG@10: 0.0578
Cov:      0.9312



Evaluating ALS: 100%|██████████| 16356/16356 [00:02<00:00, 8022.08it/s]


--- ALS ---
HR@10:   0.0284
MRR@10:  0.0117
NDCG@10: 0.0156
Cov:      0.0679

